In [1]:
#Feature engineering experiments

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

#Load data
X_train = pd.read_csv('../data/X_train_9xQjqvZ.csv', index_col = 'ROW_ID')
y_train = pd.read_csv('../data/y_train_Ppwhaz8.csv', index_col = 'ROW_ID')
X_test = pd.read_csv('../data/X_test_1zTtEnD.csv', index_col = 'ROW_ID')

#Convert target to binary
y = (y_train['target'] > 0).astype(int)

print(f"Data loaded: {X_train.shape[0]:,} training examples")
print(f"Target distribution: {y.value_counts().to_dict()}")

Data loaded: 527,073 training examples
Target distribution: {1: 267323, 0: 259750}


In [ ]:
# Experiment: Keep vs drop SIGNED_VOLUME_1

#Drop SIGNED_VOLUME_1
X_drop = X_train.drop('SIGNED_VOLUME_1', axis = 1).fillna(0) #axis 1 is column and 0 is row
feature_cols_xdrop = [col for col in X_drop.columns if col not in ['TS', 'ALLOCATION']] #loop through all column names excluding TS and ALLOCATION (they're labels not features) and put the names in a list

#Keep SIGNED_VOLUME_1 and fill (ALL including SV1) missing with 0 (like benchmark model)
X_keep = X_train.fillna(0)
feature_cols_xkeep = [col for col in X_keep.columns if col not in ['TS', 'ALLOCATION']]#loop through all column names excluding TS and ALLOCATION (they're labels not features) and put the names in a list

print(f"Drop SV1: {len(feature_cols_xdrop)} features")
print(f"Keep SV1: {len(feature_cols_xkeep)} features")

Drop SV1: 41 features
Keep SV1: 42 features


In [ ]:
# Quick 3-fold CV to compare (faster than 5-fold for testing)
def quick_cv_test(X_features, y, feature_cols, model_name):
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_features): #train_idx = row numbers for training, val_idx = row numbers for validation
        X_train_fold = X_features.iloc[train_idx][feature_cols] #select rows using index position(.iloc[train_idx]) and only the feature columns (not labels like TS and ALLOCATION)([feature_cols])
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X_features.iloc[val_idx][feature_cols]
        y_val_fold = y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1) # verbose = -1 aka don't print training progress
        model.fit(X_train_fold, y_train_fold)
        pred = model.predict(X_val_fold)
        scores.append(accuracy_score(y_val_fold, pred))
    
    print(f"{model_name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    return np.mean(scores)

print("Testing dropping vs. keeping SV1\n")
score_v1 = quick_cv_test(X_drop, y, feature_cols_xdrop, "Drop SV1")
score_v2 = quick_cv_test(X_keep, y, feature_cols_xkeep, "Keep SV1")

if score_v1 > score_v2:
    print("\n Dropping SV1 is better")
    X_base = X_drop
    feature_cols = feature_cols_xdrop
else:
    print("\n Keeping SV1 is better")
    X_base = X_keep
    feature_cols = feature_cols_xkeep

Testing dropping vs. keeping SV1

Drop SV_1: 0.5349 ± 0.0012
Keep SV_1: 0.5353 ± 0.0013

 Keeping SV1 is better


In [ ]:
# Feature engineering

# Making a copy of X_test to work with
X_eng = X_base.copy()
X_test_eng = X_test.fillna(0).copy()

# Base features; list to avoid having to type out 20 column names manually
RET_features = [f'RET_{i}' for i in range(1, 21)]
SV_features = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]

print(f"Starting with {X_eng.shape[1]} colunmns")

Starting with 44 colunmns


In [9]:
# Feature 1: Moving Averages of Returns (short-term vs. long-term performance)

for window in [3, 5, 10, 15, 20]:
    X_eng[f'RET_AVG_{window}'] = X_eng[RET_features[:window]].mean(axis= 1)
    X_test_eng[f'RET_AVG_{window}'] = X_test_eng[RET_features[:window]].mean(axis= 1)

print("Created moving average features:")
print([f'RET_AVG_{w}' for w in [3, 5, 10, 15, 20]])
print(f"\nNow have {X_eng.shape[1]} columns")

Created moving average features:
['RET_AVG_3', 'RET_AVG_5', 'RET_AVG_10', 'RET_AVG_15', 'RET_AVG_20']

Now have 52 columns


In [8]:
# Feature 2: Volatility (Standard Deviation) (How risky or stable this allocation is)

for window in [5, 10, 20]:
    X_eng[f'RET_STD_{window}'] = X_eng[RET_features[:window]].std(axis=1)
    X_test_eng[f'RET_STD_{window}'] = X_test_eng[RET_features[:window]].std(axis=1)

print("Created volatility features:")
print([f'RET_STD_{w}' for w in [5, 10, 20]])
print(f"Now have {X_eng.shape[1]} columns")


Created volatility features:
['RET_STD_5', 'RET_STD_10', 'RET_STD_20']
Now have 52 columns


In [10]:
# Feature 3: Momentum (Recent vs Older Performance)(Is the allocation getting better or worse)

# Recent (last 3 days) minus older (days 15-20)
X_eng['MOMENTUM_SHORT'] = X_eng['RET_AVG_3'] - X_eng[RET_features[15:20]].mean(axis=1)

# Recent (last 5 days) minus full average
X_eng['MOMENTUM_LONG'] = X_eng['RET_AVG_5'] - X_eng['RET_AVG_20']

# Same for test
X_test_eng['MOMENTUM_SHORT'] = X_test_eng['RET_AVG_3'] - X_test_eng[RET_features[15:20]].mean(axis=1)
X_test_eng['MOMENTUM_LONG'] = X_test_eng['RET_AVG_5'] - X_test_eng['RET_AVG_20']

print("Created momentum features: MOMENTUM_SHORT, MOMENTUM_LONG")
print(f"Now have {X_eng.shape[1]} columns")

Created momentum features: MOMENTUM_SHORT, MOMENTUM_LONG
Now have 54 columns


In [12]:
# Feature 4: Cross-Allocation Features (How is this allocation performing compared to others on the same day)

# For each time window, calculate the average performance of ALL allocations on that date
for window in [5, 10, 20]:
    # Average performance of all allocations on this date
    X_eng[f'DATE_AVG_RET_{window}'] = X_eng.groupby('TS')[f'RET_AVG_{window}'].transform('mean')
    X_test_eng[f'DATE_AVG_RET_{window}'] = X_test_eng.groupby('TS')[f'RET_AVG_{window}'].transform('mean')
    
    # How much better/worse is this allocation vs. the average?
    X_eng[f'RELATIVE_PERF_{window}'] = X_eng[f'RET_AVG_{window}'] - X_eng[f'DATE_AVG_RET_{window}']
    X_test_eng[f'RELATIVE_PERF_{window}'] = X_test_eng[f'RET_AVG_{window}'] - X_test_eng[f'DATE_AVG_RET_{window}']

print("Created cross-allocation features:")
print([f'RELATIVE_PERF_{w}' for w in [5, 10, 20]])
print(f"Now have {X_eng.shape[1]} columns")

Created cross-allocation features:
['RELATIVE_PERF_5', 'RELATIVE_PERF_10', 'RELATIVE_PERF_20']
Now have 60 columns


In [13]:
# Feature 5: Trend Direction (Is there a clear upward or downward trend in recent performance)

# Count how many of last 5 days were positive
X_eng['POSITIVE_DAYS_5'] = (X_eng[RET_features[:5]] > 0).sum(axis=1)
X_test_eng['POSITIVE_DAYS_5'] = (X_test_eng[RET_features[:5]] > 0).sum(axis=1)

# Count how many of last 10 days were positive  
X_eng['POSITIVE_DAYS_10'] = (X_eng[RET_features[:10]] > 0).sum(axis=1)
X_test_eng['POSITIVE_DAYS_10'] = (X_test_eng[RET_features[:10]] > 0).sum(axis=1)

print("Created trend features: POSITIVE_DAYS_5, POSITIVE_DAYS_10")
print(f"\nFinal feature count: {X_eng.shape[1]} columns")

# Summary of all new features
new_features = [col for col in X_eng.columns if col not in X_base.columns]
print(f"\nCreated {len(new_features)} new features:")
print(new_features)

Created trend features: POSITIVE_DAYS_5, POSITIVE_DAYS_10

Final feature count: 62 columns

Created 18 new features:
['RET_AVG_3', 'RET_AVG_5', 'RET_AVG_10', 'RET_AVG_15', 'RET_AVG_20', 'RET_STD_5', 'RET_STD_10', 'RET_STD_20', 'MOMENTUM_SHORT', 'MOMENTUM_LONG', 'DATE_AVG_RET_5', 'RELATIVE_PERF_5', 'DATE_AVG_RET_10', 'RELATIVE_PERF_10', 'DATE_AVG_RET_20', 'RELATIVE_PERF_20', 'POSITIVE_DAYS_5', 'POSITIVE_DAYS_10']


In [14]:
# Prepare feature lists
base_features = [col for col in X_base.columns if col not in ['TS', 'ALLOCATION']]
eng_features = [col for col in X_eng.columns if col not in ['TS', 'ALLOCATION']]

print(f"Baseline features: {len(base_features)}")
print(f"Engineered features: {len(eng_features)}")
print(f"New features added: {len(eng_features) - len(base_features)}")

# Test both with 5-fold CV
def test_features(X_data, feature_list, name):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    print(f"\nTesting {name}:")
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_data), 1):
        X_train_fold = X_data.iloc[train_idx][feature_list]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X_data.iloc[val_idx][feature_list]
        y_val_fold = y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=100, 
            learning_rate=0.05, 
            random_state=42, 
            verbose=-1
        )
        model.fit(X_train_fold, y_train_fold)
        pred = model.predict(X_val_fold)
        acc = accuracy_score(y_val_fold, pred)
        scores.append(acc)
        print(f"  Fold {fold}: {acc:.4f}")
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    print(f"  Mean: {mean_score:.4f} ± {std_score:.4f}")
    return mean_score

print("="*50)
baseline_score = test_features(X_base, base_features, "BASELINE (no feature engineering)")
print("="*50)
engineered_score = test_features(X_eng, eng_features, "ENGINEERED FEATURES")
print("="*50)

# Compare
improvement = (engineered_score - baseline_score) * 100
print(f"\n{'='*50}")
print(f"RESULTS:")
print(f"Baseline:    {baseline_score:.4f}")
print(f"Engineered:  {engineered_score:.4f}")
print(f"Improvement: {improvement:+.2f}% ({'+' if improvement > 0 else ''}{improvement:.4f})")
print(f"{'='*50}")

if engineered_score > baseline_score:
    print("Feature engineering helped")
else:
    print("Feature engineering didn't help (might be overfitting)")

Baseline features: 42
Engineered features: 60
New features added: 18

Testing BASELINE (no feature engineering):
  Fold 1: 0.5398
  Fold 2: 0.5339
  Fold 3: 0.5366
  Fold 4: 0.5363
  Fold 5: 0.5359
  Mean: 0.5365 ± 0.0019

Testing ENGINEERED FEATURES:
  Fold 1: 0.5480
  Fold 2: 0.5448
  Fold 3: 0.5458
  Fold 4: 0.5456
  Fold 5: 0.5461
  Mean: 0.5461 ± 0.0011

RESULTS:
Baseline:    0.5365
Engineered:  0.5461
Improvement: +0.96% (+0.9568)
Feature engineering helped


In [15]:
# Train final model on ALL training data with engineered features
print("Training final model on all data with engineered features:")

final_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

final_model.fit(X_eng[eng_features], y)

# Prepare test data (same features)
X_test_final = X_test_eng[eng_features]

# Predict
test_predictions = final_model.predict(X_test_final)

# Create submission
submission = pd.DataFrame({
    'ROW_ID': X_test.index,
    'prediction': test_predictions
})

submission.to_csv('../data/secondFeatEng_sub.csv', index=False)

print(f"Submission created")
print(f"Predictions: {submission['prediction'].value_counts().to_dict()}")
print(f"\nCV Score: {engineered_score:.4f}")
print(f"Previous leaderboard: 0.5090")

Training final model on all data with engineered features:
Submission created
Predictions: {1: 18635, 0: 13235}

CV Score: 0.5461
Previous leaderboard: 0.5090


In [16]:
# Hyperparameter tuning - test different configurations

from sklearn.model_selection import KFold

configs = [
    {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': -1, 'name': 'baseline'},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': -1, 'name': 'more_trees'},
    {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': -1, 'name': 'lower_lr'},
    {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': -1, 'name': 'balanced'},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 7, 'name': 'depth_limit'},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 5, 'name': 'shallow_trees'},
]

results = []

print("Testing different hyperparameter configurations:")
print("="*60)

for config in configs:
    name = config.pop('name')
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_eng):
        X_train_fold = X_eng.iloc[train_idx][eng_features]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X_eng.iloc[val_idx][eng_features]
        y_val_fold = y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**config, random_state=42, verbose=-1)
        model.fit(X_train_fold, y_train_fold)
        pred = model.predict(X_val_fold)
        scores.append(accuracy_score(y_val_fold, pred))
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    results.append({
        'name': name,
        'config': config,
        'cv_score': mean_score,
        'std': std_score
    })
    
    print(f"{name:15s}: {mean_score:.4f} ± {std_score:.4f}")

print("="*60)

# Find best config
best = max(results, key=lambda x: x['cv_score'])
print(f"\nBest configuration: {best['name']}")
print(f"   Score: {best['cv_score']:.4f}")
print(f"   Config: {best['config']}")

Testing different hyperparameter configurations:
baseline       : 0.5461 ± 0.0011
more_trees     : 0.5551 ± 0.0008
lower_lr       : 0.5302 ± 0.0016
balanced       : 0.5544 ± 0.0015
depth_limit    : 0.5530 ± 0.0013
shallow_trees  : 0.5481 ± 0.0015

Best configuration: more_trees
   Score: 0.5551
   Config: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': -1}


In [18]:
# Fine-tune around the winner - more trees (200)
fine_tune_configs = [
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': -1, 'name': 'winner'},
    {'n_estimators': 250, 'learning_rate': 0.05, 'max_depth': -1, 'name': 'even_more_trees'},
    {'n_estimators': 200, 'learning_rate': 0.07, 'max_depth': -1, 'name': 'higher_lr'},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'name': 'regularized'},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 10, 'name': 'depth_10'},
]

fine_results = []
print("Fine-tuning around best configuration:")
print("="*60)

for config in fine_tune_configs:
    name = config.pop('name')
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_eng):
        X_train_fold = X_eng.iloc[train_idx][eng_features]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X_eng.iloc[val_idx][eng_features]
        y_val_fold = y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**config, random_state=42, verbose=-1)
        model.fit(X_train_fold, y_train_fold)
        pred = model.predict(X_val_fold)
        scores.append(accuracy_score(y_val_fold, pred))
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    fine_results.append({
        'name': name,
        'config': config,
        'cv_score': mean_score,
        'std': std_score
    })
    
    print(f"{name:20s}: {mean_score:.4f} ± {std_score:.4f}")

print("="*60)

# Overall best
all_results = results + fine_results
overall_best = max(all_results, key=lambda x: x['cv_score'])
print(f"\nOverall best: {overall_best['name']}")
print(f"   CV Score: {overall_best['cv_score']:.4f}")
print(f"   Config: {overall_best['config']}")

Fine-tuning around best configuration:
winner              : 0.5551 ± 0.0008
even_more_trees     : 0.5575 ± 0.0011
higher_lr           : 0.5592 ± 0.0013
regularized         : 0.5551 ± 0.0008
depth_10            : 0.5545 ± 0.0016

Overall best: higher_lr
   CV Score: 0.5592
   Config: {'n_estimators': 200, 'learning_rate': 0.07, 'max_depth': -1}


In [19]:
# Train with best hyperparameters on all data
print("Training final optimized model:")

best_config = {
    'n_estimators': 200,
    'learning_rate': 0.07,
    'max_depth': -1,
    'random_state': 42,
    'verbose': -1
}

final_model = lgb.LGBMClassifier(**best_config)
final_model.fit(X_eng[eng_features], y)

# Predict on test
test_predictions = final_model.predict(X_test_eng[eng_features])

# Create submission
submission = pd.DataFrame({
    'ROW_ID': X_test.index,
    'prediction': test_predictions
})

submission.to_csv('../data/secondFeatEng_subOpt.csv', index=False)

print("Optimised submission created")
print(f"\nCV Score: {overall_best['cv_score']:.4f} (55.92%)")
print(f"Previous leaderboard: 0.5118 (51.18%)")
print(f"Expected improvement: ~0.3-0.5%")
print(f"\nPrediction distribution:")
print(submission['prediction'].value_counts())

Training final optimized model:
Optimised submission created

CV Score: 0.5592 (55.92%)
Previous leaderboard: 0.5118 (51.18%)
Expected improvement: ~0.3-0.5%

Prediction distribution:
prediction
1    18050
0    13820
Name: count, dtype: int64
